# Airbnb Price Analytics — City C Transfer

**Region:** United Kingdom
**Currency:** GBP

This notebook records the first run of the frozen model on the third city (Greater Manchester) and diagnoses its errors against the London and Bristol test rows.

It only reads `workspace.airbnb_gold.predictions` and `workspace.airbnb_gold.features`. Nothing is trained, scored or written.

## 1. Configuration

The third city, model version, feature version, catalog and schema are notebook parameters. There is no `write_tables` switch, because the notebook writes nothing.

In [ ]:
import re
import numpy as np
import pandas as pd
from pyspark.sql import functions as F
dbutils.widgets.text('catalog', 'workspace')
dbutils.widgets.text('gold_schema', 'airbnb_gold')
dbutils.widgets.text('city', 'greater-manchester')
dbutils.widgets.text('model_version', 'v1')
dbutils.widgets.text('feature_version', 'v1')

Widget values are read once and validated before they are used in table names and filters.

In [ ]:
catalog = dbutils.widgets.get('catalog').strip()
gold_schema = dbutils.widgets.get('gold_schema').strip()
city = dbutils.widgets.get('city').strip().lower()
model_version = dbutils.widgets.get('model_version').strip()
feature_version = dbutils.widgets.get('feature_version').strip()
for name, value in [('catalog', catalog), ('gold_schema', gold_schema)]:
    if not re.fullmatch('[A-Za-z][A-Za-z0-9_]*', value):
        raise ValueError(f'Invalid {name} identifier')
if not re.fullmatch('[a-z][a-z0-9_-]*', city):
    raise ValueError(f'Invalid city: {city!r}')

Table names and constants. `MIN_CATEGORY_COUNT` must equal `min_category_count` in the model manifest: property types with fewer training rows were grouped as `Other` by the model.

In [ ]:
KEY_COLUMNS = ['city', 'listing_id', 'snapshot_date']
FEATURES = ['room_type', 'property_type', 'accommodates', 'bedrooms', 'beds', 'bathrooms',
            'latitude', 'longitude', 'minimum_nights', 'maximum_nights']
MIN_CATEGORY_COUNT = 20
features_table = f'{catalog}.{gold_schema}.features'
predictions_table = f'{catalog}.{gold_schema}.predictions'
print(f'city={city}, model_version={model_version}, feature_version={feature_version}')

## 2. Load

Two groups of predictions are compared, both from the same model version:

- third city: the `first_run` predictions (`split = 'holdout'`);
- London and Bristol: the test rows of the development run (`run_label = 'development'`, `split = 'test'`), written by `03_predict` for each development city.

In [ ]:
predictions = (spark.table(predictions_table)
               .filter(F.col('model_version') == model_version)
               .filter(((F.col('city') == city) & (F.col('run_label') == 'first_run') & (F.col('split') == 'holdout'))
                       | ((F.col('run_label') == 'development') & (F.col('split') == 'test'))))

The model inputs come from the features table of the same feature version. Every prediction must have exactly one feature row.

In [ ]:
features = spark.table(features_table).filter(F.col('feature_version') == feature_version)
rows = predictions.join(features.select(*KEY_COLUMNS, *FEATURES, F.lit(True).alias('has_features')), KEY_COLUMNS, 'left')
pdf = rows.toPandas()
assert not pdf.duplicated(KEY_COLUMNS).any(), 'Duplicate prediction or feature rows'
assert pdf['has_features'].notna().all(), 'Some predictions have no feature row'
assert (pdf['city'] == city).any(), f'No first_run predictions for {city} and model {model_version}'
assert (pdf['split'] == 'test').any(), 'No development test predictions: run 03_predict for london and bristol first'
print(pdf.groupby('city').size().to_dict())

Training rows are used for two references only: the property types the model knows (at least `MIN_CATEGORY_COUNT` training rows, as in `RareCategoryGrouper`) and the coordinate range seen in training.

In [ ]:
train = features.filter(F.col('split') == 'train')
type_counts = train.groupBy('property_type').count().toPandas()
known_types = set(type_counts.loc[type_counts['count'] >= MIN_CATEGORY_COUNT, 'property_type'].dropna())
print(f'{len(known_types)} property types known to the model')

Errors are computed for evaluated rows only: a valid target and a valid prediction. Signed error is `predicted - actual`, so a positive value means the price was overestimated.

In [ ]:
pdf['evaluated'] = pdf['actual_price_gbp'].notna() & (pdf['prediction_status'] == 'valid')
ev = pdf[pdf['evaluated']].copy()
for predictor in ['model', 'baseline']:
    column = 'predicted_price_gbp' if predictor == 'model' else 'baseline_price_gbp'
    ev[f'{predictor}_error'] = ev[column] - ev['actual_price_gbp']
    ev[f'{predictor}_ae'] = ev[f'{predictor}_error'].abs()
ev['property_group'] = np.where(ev['property_type'].isin(known_types), 'known', 'Other')

## 3. First Run (Unchanged)

The first third-city run used the frozen `v1` artifacts without any change: `03_predict` with `run_label = first_run` and `write_tables = true`. It completed without errors; no step failed and no listing failed to score.

| Item | Third city (first run) | London + Bristol test |
| --- | --- | --- |
| Rows loaded | 7,799 | — |
| Listings with a valid price | 6,692 | 12,434 |
| Prediction coverage | 100% | 100% |
| Baseline MAE | £100.84 | £104.17 |
| Baseline median AE | £73.50 | £44.84 |
| Model MAE | £79.68 | £76.93 |
| Model median AE | £43.38 | £32.31 |

- Baseline fallback: 7,797 listings used the room-type and capacity group, 2 the room-type median.
- 84 listings had a property type outside the model's categories and were mapped to `Other`, for example "Shared room in home" (14), "Barn" (9), "Private room in cottage" (9), "Entire vacation home" (8) and "Camper/RV" (7).
- Missing inputs: `beds` 210, `minimum_nights` and `maximum_nights` 32 each, `bathrooms` 15, `bedrooms` 5. The model handles missing numbers natively.

The input checksum and run time are stored with the first-run metrics in `model_metrics` (`dataset = 'city_c_first_run'`).

The recorded values are recomputed from the stored predictions and features. Any difference stops the notebook, because the tables would then no longer hold the first-run evidence.

In [ ]:
RECORDED = {'rows_loaded': 7799, 'valid_prices': 6692, 'prediction_coverage': 1.0,
            'baseline_mae': 100.84, 'baseline_median_ae': 73.50, 'model_mae': 79.68, 'model_median_ae': 43.38,
            'fallback_group': 7797, 'fallback_room_type': 2, 'property_type_other': 84,
            'missing_beds': 210, 'missing_minimum_nights': 32, 'missing_maximum_nights': 32,
            'missing_bathrooms': 15, 'missing_bedrooms': 5}

Recomputed values for the third city. Metrics are rounded to pence, as recorded.

In [ ]:
c = pdf[pdf['city'] == city]
c_ev = ev[ev['city'] == city]
recomputed = {'rows_loaded': len(c), 'valid_prices': int(c['actual_price_gbp'].notna().sum()),
              'prediction_coverage': c_ev.shape[0] / c['actual_price_gbp'].notna().sum(),
              'baseline_mae': c_ev['baseline_ae'].mean(), 'baseline_median_ae': c_ev['baseline_ae'].median(),
              'model_mae': c_ev['model_ae'].mean(), 'model_median_ae': c_ev['model_ae'].median(),
              'fallback_group': int((c['baseline_level'] == 'group').sum()),
              'fallback_room_type': int((c['baseline_level'] == 'room_type').sum()),
              'property_type_other': int((~c['property_type'].isin(known_types)).sum())}
for column in ['beds', 'minimum_nights', 'maximum_nights', 'bathrooms', 'bedrooms']:
    recomputed[f'missing_{column}'] = int(c[column].isna().sum())

Recorded and recomputed values side by side.

In [ ]:
check = pd.DataFrame({'recorded': RECORDED, 'recomputed': pd.Series(recomputed).astype(float).round(2)})
check['matches'] = (check['recorded'] - check['recomputed']).abs() < 0.006
display(check.reset_index().rename(columns={'index': 'item'}))
assert check['matches'].all(), f"Stored first run differs from the record: {list(check.index[~check['matches']])}"

## 4. Diagnosis

All comparisons use evaluated rows: third-city first-run rows against London and Bristol test rows. The same summary function is used for every breakdown.

In [ ]:
def error_summary(frame, by):
    return (frame.groupby(by, dropna=False)
                 .agg(listings=('listing_id', 'size'),
                      baseline_mae=('baseline_ae', 'mean'), model_mae=('model_ae', 'mean'),
                      baseline_median_ae=('baseline_ae', 'median'), model_median_ae=('model_ae', 'median'))
                 .round(2).reset_index())

### 4.1 Price Distribution

Actual nightly prices per city. A different price level from the training cities shows up here before any model error.

In [ ]:
price = ev.groupby('city')['actual_price_gbp']
display(pd.DataFrame({'listings': price.size(), 'q25': price.quantile(0.25), 'median': price.median(),
                      'q75': price.quantile(0.75), 'q90': price.quantile(0.90), 'mean': price.mean()})
        .round(2).reset_index())

### 4.2 Bias

Mean and median signed error (`predicted - actual`) per city. A positive median means the predictor overprices the typical listing; the share overpriced shows how often. The baseline is included because it shows the pure price-level effect of using London and Bristol medians.

In [ ]:
bias = ev.groupby('city').agg(
    listings=('listing_id', 'size'),
    model_mean_error=('model_error', 'mean'), model_median_error=('model_error', 'median'),
    model_share_overpriced=('model_error', lambda s: (s > 0).mean()),
    baseline_mean_error=('baseline_error', 'mean'), baseline_median_error=('baseline_error', 'median'),
    baseline_share_overpriced=('baseline_error', lambda s: (s > 0).mean()))
display(bias.round(3).reset_index())

### 4.3 Error by Room Type and Capacity

MAE and median absolute error by room type, per city.

In [ ]:
display(error_summary(ev, ['room_type', 'city']))

Capacity is grouped into bands, so that each row has enough listings: 1, 2, 3–4, 5–6 and 7 or more guests.

In [ ]:
ev['capacity'] = pd.cut(ev['accommodates'], bins=[0, 1, 2, 4, 6, np.inf], labels=['1', '2', '3-4', '5-6', '7+']).astype(str)
display(error_summary(ev, ['capacity', 'city']))

Median signed model error by capacity band shows whether any bias grows with listing size.

In [ ]:
display(ev.pivot_table(index='capacity', columns='city', values='model_error', aggfunc='median').round(2).reset_index())

### 4.4 Coordinates

Latitude and longitude ranges per city: training rows for London and Bristol, all first-run rows for the third city.

In [ ]:
coordinates = features.filter((F.col('split') == 'train') | (F.col('city') == city))
display(coordinates.groupBy('city').agg(
    F.count('*').alias('listings'),
    F.round(F.min('latitude'), 3).alias('min_latitude'), F.round(F.max('latitude'), 3).alias('max_latitude'),
    F.round(F.min('longitude'), 3).alias('min_longitude'), F.round(F.max('longitude'), 3).alias('max_longitude')))

Third-city listings outside the pooled training range. The model is a tree ensemble: a coordinate beyond the training range is treated like the nearest edge of that range, so these listings are placed at the northern, southern, eastern or western edge of London and Bristol.

In [ ]:
bounds = train.agg(F.min('latitude'), F.max('latitude'), F.min('longitude'), F.max('longitude')).first()
lat_out = c['latitude'].notna() & ~c['latitude'].between(bounds[0], bounds[1])
lon_out = c['longitude'].notna() & ~c['longitude'].between(bounds[2], bounds[3])
display(pd.DataFrame([{'listings': len(c), 'latitude_outside': int(lat_out.sum()), 'longitude_outside': int(lon_out.sum()),
                       'either_outside': int((lat_out | lon_out).sum()), 'share_either_outside': round((lat_out | lon_out).mean(), 3)}]))

### 4.5 Known and `Other` Property Types

Error for property types the model knows against types it mapped to `Other` (rare, unseen or missing), per city.

In [ ]:
display(error_summary(ev, ['property_group', 'city']))

The most frequent `Other` property types in the third city, with their error. Counts only; no listing identifiers.

In [ ]:
other_types = error_summary(ev[(ev['city'] == city) & (ev['property_group'] == 'Other')], ['property_type'])
display(other_types.sort_values('listings', ascending=False).head(15))

## 5. Findings and Changes

Findings from section 4 (third-city first run against London and Bristol test rows):

- **Price level.** The third city is cheaper: median price £117, against £134 in Bristol and £176 in London.
- **Bias.** The model overprices the third city: median signed error +£31.80, with 78.3% of listings overpriced. In London (+£4.64, 54.7%) and Bristol (+£1.41, 51.7%) the model is close to unbiased.
- **Capacity.** In the third city the overpricing grows with capacity: +£12.55 for 1 guest, +£13.98 for 2, +£41.56 for 3–4, +£76.42 for 5–6 and +£132.38 for 7 or more. In London and Bristol the median signed error stays within about £11 in every band.
- **Coordinates.** Every third-city listing is outside the training latitude range (53.336–53.665 against a training maximum of 51.681). Its longitudes lie mostly inside Bristol's range, so the model sees a combination of latitude and longitude that does not occur in training.
- **Property types.** Only 78 evaluated listings have an `Other` property type, and their error is similar to that of known types. This is not the main problem.
- The recomputed first-run values match the record in section 3.

Each candidate change is classified as required by the holdout protocol: a **pipeline fix** corrects how data are ingested or cleaned; a **model change** alters the features, preprocessing, training or model. The first-run results above are never overwritten.

| # | Change | Type | Reason | Decision |
| --- | --- | --- | --- | --- |
| 1 | Check the parsing of `beds`, `minimum_nights`, `maximum_nights`, `bathrooms` and `bedrooms` for the third city; fix it if values are present in the source but lost by the ETL. | pipeline fix | 210 listings lack `beds`, but missing inputs do not explain a bias that grows with capacity. | Not implemented; to check with the ETL owner. |
| 2 | Flag or exclude placeholder-like prices (around £5,000 or £10,000 a night) in the cleaning rules for all cities. | pipeline fix | They dominate the largest A/B test errors, but the third-city problem is a shift of typical listings (median error), not a few outliers. | Not implemented. |
| 3 | Replace raw `latitude`/`longitude` with the distance to the city centre (median coordinates of the city's listings, no prices). | model change | Raw coordinates cannot describe a city outside the training area; all third-city latitudes are outside the training range. | **Tested (`v2`), rejected by cross-validation.** |
| 4 | Group property types into broader families before the rare-category step. | model change | Few `Other` listings (78 evaluated) and an error similar to known types. | Rejected without testing. |
| 5 | Balance London and Bristol during training (sample weights per city). | model change | London provides 96.4% of training rows; the third city is priced closer to Bristol than to London. | **Tested with change 3, rejected by cross-validation.** |
| 6 | Predict a city-relative price and rescale with the city's median price. | model change | Would address the level difference directly, but needs a third-city price level from third-city prices or an external source. | Not implemented; it would use holdout information. |

### Model change test (`v2`)

Changes 3 and 5 were tested as candidate model `v2` in the training notebook, with the same split, baseline and selection rule as `v1`: pooled 5-fold cross-validated MAE in GBP on London and Bristol training rows only. The A/B test set and third-city prices were not used, and `v2` was not frozen.

| candidate | bristol | london | combined |
| --- | --- | --- | --- |
| **`v1` (frozen): coordinates, HGB lr=0.05 leaves=63** | **41.20** | **85.70** | **84.08** |
| `v2` distance to centre, best HGB (lr=0.1 leaves=63) | 106.74 | 90.93 | 91.51 |
| `v2` distance to centre, other unweighted HGB settings | — | — | 91.77–91.94 |
| `v2` distance to centre, city-weighted HGB (4 settings) | 53.19–55.58 | 99.68–101.41 | 98.07–99.66 |
| `v2` distance to centre, Ridge | 107.30 | 101.86 | 102.06 |

No `v2` variant beats `v1`, so the model change is rejected and `v1` remains the final model. No `after_model_change` run of the third city was made.

**Why `v2` fails.** `city` is not a feature, so in `v1` the coordinates also tell the model which city a listing is in, and therefore its price level. A distance to the centre has the same meaning in every city and removes that information: Bristol listings are priced like London listings at the same distance, and Bristol's error rises from £41.20 to £106.74. London loses less (£85.70 to £90.93), because distance keeps part of the within-city location effect. City weighting gives Bristol more influence and lowers its error to about £53–56, but only by making London worse (about £100–101), and the pooled MAE is still well above `v1`.

**What this means for the third city.** The test supports the diagnosis: the model takes a city's price level from its coordinates. The third city lies outside the training coordinates, so the model cannot know its lower price level and overprices it, increasingly so for larger listings. A model without coordinates cannot know any city's level either. Correcting the level needs information about the third city's prices (change 6), which the unseen-city design excludes.

The baseline is not changed: it is the required comparator and stays the same for all runs.